# Sound Bubble — Colab training + ONNX export

This notebook assumes the Drive-synced bridge at
`MyDrive/<...>/CDE3301/colab_bridge/` already contains:

- `code/` — rsync'd snapshot from `sync_workspace_to_bridge.ps1`
- `datasets/` — tarballs (`syn_1m.tar`, `syn_1_5m.tar`, `syn_2m.tar`, `syn_test.tar`)

On completion it writes:

- `runs/<exp_name>/` — trained run directory + checkpoints
- `zoo/model_<radius>m.onnx` + `zoo/model_<radius>m.runtime.json`

Back on the PC, `sync_bridge_to_workspace.ps1` pulls the zoo into `edge/zoo/`.

**Runtime type:** T4 GPU (Runtime → Change runtime type → T4 GPU).

### Which training track is this?

This notebook drives the **single-distance `_optim` track** — one model per radius,
using the `raspberrypi_local_pretrain*.json` configs. The exporter will still stamp
full provenance (epochs, source_config, source_run_dir, etc.) into each
`model_<radius>m.runtime.json`.

The **distance-aware `_dis_embd3` track** (one model covers 1.0/1.5/2.0 m via a
runtime `dis_embed` input) uses a different config
(`TFG_S_big_newdis_v3_pt_fix_MutiLoss/config.json` in the local workspace) and was
already trained on HPC. The updated exporter (`edge/export_to_onnx.py`) auto-detects
that architecture and produces a distance-aware ONNX — point it at the run_dir and
it Just Works locally; no Colab round-trip required unless you're continuing training.
If you do want to continue-train that model on Colab, copy its config + init checkpoint
into `bridge/` and adapt cell 1 below accordingly.

In [ ]:
# ==== User knobs — set these per run ====
RADIUS = 1.5                   # one of 1.0 / 1.5 / 2.0
EPOCHS_TARGET = 150            # overrides config's n_epochs
RESUME_FROM_DRIVE = True       # continue from the latest bridge run if one exists
EXPERIMENT_TAG = ""            # optional suffix, e.g. "colab_t4_run3"

# Force a specific folder name under bridge/runs/ (rare). Leave empty for auto-pick.
EXP_NAME_OVERRIDE = ""

# ==== Bridge path — edit the MyDrive subpath once if yours differs ====
BRIDGE = "/content/drive/MyDrive/Obsidian Notes/CDE3301/colab_bridge"

# ==== Derived ====
CONFIG_NAME_BY_RADIUS = {
    1.0: "raspberrypi_local_pretrain_1m.json",
    1.5: "raspberrypi_local_pretrain.json",
    2.0: "raspberrypi_local_pretrain_2m.json",
}
CONFIG_NAME = CONFIG_NAME_BY_RADIUS[RADIUS]
# Human-readable tag for filenames (keeps a dot when present, e.g. 1.5m)
RADIUS_TAG = f"{RADIUS:g}m"

# EXP_NAME is finalized in the next cell *after* Drive is mounted, because we scan
# bridge/runs/ for an existing checkpoint folder to resume into.
EXP_NAME = None

print(f"[cfg] radius={RADIUS_TAG} config={CONFIG_NAME} epochs_target={EPOCHS_TARGET} (mount Drive next to pick exp_name)")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, pathlib

assert os.path.isdir(BRIDGE), f"Bridge not found at {BRIDGE}. Check MyDrive path in the cell above."
for sub in ("code", "datasets", "runs", "zoo"):
    pathlib.Path(BRIDGE, sub).mkdir(parents=True, exist_ok=True)

# ---- Resolve EXP_NAME (auto-resume folder pick) ----
radius_slug = f"{RADIUS:g}m".replace(".", "_")
default_exp = f"optim_pretrain_{radius_slug}" + (f"_{EXPERIMENT_TAG}" if EXPERIMENT_TAG else "")

if EXP_NAME_OVERRIDE:
    EXP_NAME = EXP_NAME_OVERRIDE
    print(f"[cfg] exp_name override -> {EXP_NAME}")
else:
    runs_root = os.path.join(BRIDGE, "runs")
    candidates = []

    if RESUME_FROM_DRIVE and os.path.isdir(runs_root):
        def _resume_mtime(name: str) -> float:
            ckpt_dir = os.path.join(runs_root, name, "checkpoints")
            mtimes = []
            for fn in ("last.pt", "best.pt"):
                fp = os.path.join(ckpt_dir, fn)
                if os.path.isfile(fp):
                    try:
                        mtimes.append(os.path.getmtime(fp))
                    except OSError:
                        pass
            return max(mtimes) if mtimes else 0.0

        # Prefer exact default name if it already exists with a resume checkpoint.
        default_ckpt_dir = os.path.join(runs_root, default_exp, "checkpoints")
        default_last = os.path.join(default_ckpt_dir, "last.pt")
        default_best = os.path.join(default_ckpt_dir, "best.pt")
        if os.path.isfile(default_last) or os.path.isfile(default_best):
            candidates.append(default_exp)

        # Otherwise scan for historical naming patterns for this radius.
        # Use explicit suffix tokens (avoid regex false-positives like matching 1.5 inside 11.5).
        dot_tag = f"{RADIUS:g}m"          # e.g. 1.5m
        us_tag = f"{radius_slug}m"      # e.g. 1_5m
        alt_tags = {dot_tag, us_tag}
        if float(RADIUS).is_integer():
            alt_tags.add(f"{int(float(RADIUS))}m")  # e.g. 2m for RADIUS=2.0

        def _matches_radius_folder(name: str) -> bool:
            if name == default_exp:
                return False
            # optim_pretrain_* tokens
            if "optim_pretrain" in name:
                return any(t in name for t in alt_tags)
            # legacy run_* tokens
            if name.startswith("run_") or "_run_" in name:
                return any(t in name for t in alt_tags)
            return False

        for name in sorted(os.listdir(runs_root)):
            p = os.path.join(runs_root, name)
            if not os.path.isdir(p):
                continue
            ckpt_dir = os.path.join(p, "checkpoints")
            if not (os.path.isfile(os.path.join(ckpt_dir, "last.pt")) or os.path.isfile(os.path.join(ckpt_dir, "best.pt"))):
                continue

            if name == default_exp:
                continue

            if EXPERIMENT_TAG:
                if not (name.endswith(f"_{EXPERIMENT_TAG}") or name == f"{default_exp}_{EXPERIMENT_TAG}"):
                    # If user set a tag, don't accidentally resume an unrelated run.
                    continue

            if _matches_radius_folder(name):
                candidates.append(name)

    if RESUME_FROM_DRIVE and candidates:
        def _mtime_key(n):
            return _resume_mtime(n)

        EXP_NAME = sorted(set(candidates), key=_mtime_key)[-1]
        print(f"[cfg] auto-picked exp_name={EXP_NAME} (latest checkpoint mtime among {len(set(candidates))} matches)")
    else:
        EXP_NAME = default_exp
        if RESUME_FROM_DRIVE:
            print(f"[cfg] no resumable run found under {runs_root}; using fresh exp_name={EXP_NAME}")
        else:
            print(f"[cfg] RESUME_FROM_DRIVE=False; using fresh exp_name={EXP_NAME}")

print(f"[cfg] radius={RADIUS_TAG} config={CONFIG_NAME} epochs_target={EPOCHS_TARGET} exp_name={EXP_NAME}")
!ls -la "$BRIDGE"

In [ ]:
# Stage the code onto Colab's fast local disk. Running directly off Drive is
# viable but small-file I/O is slow; local copy is a one-time cost.
#
# IMPORTANT: the bridge snapshot lives under `code/` (repo root). We `cd` into
# that repo root so imports like `import src...` work.
WORK_ROOT = "/content/Sound_Bubble"
REPO_ROOT = f"{WORK_ROOT}/code"
!rm -rf "$WORK_ROOT"
!mkdir -p "$WORK_ROOT"
!cp -r "$BRIDGE/code" "$WORK_ROOT"
%cd "$REPO_ROOT"
!ls -la | head -30

In [ ]:
# (moved) Debug/preflight checks now run inside the training cell *after* LOCAL_RUN exists.


In [ ]:
# Install training deps. Audited against what train_pt.py actually imports
# transitively (SNRLPLoss -> asteroid; tfgridnet_causal -> asteroid_filterbanks;
# MultiResoLoss -> auraloss; dataset perturbations -> torchaudio).
#
# IMPORTANT: Colab images now often include jax/opencv stacks that require
# numpy>=2. Older onnxruntime pins (e.g. 1.18.1) require numpy<2 and will
# conflict. So we:
#   1) upgrade numpy to 2.x
#   2) install a numpy-2-compatible onnxruntime
#   3) run `pip check` to surface any remaining conflicts
!pip -q install -U "numpy>=2.0,<2.3" 2>&1 | tail -8

!pip -q install -U \
    wandb \
    soundfile librosa \
    "onnx>=1.16.2" \
    "onnxruntime>=1.20.0" \
    asteroid-filterbanks==0.4.0 \
    asteroid==0.7.0 \
    auraloss==0.4.0 \
    tqdm 2>&1 | tail -12

print("[pip] check (conflicts below mean install is NOT clean):")
!pip -q check || true

import numpy as np
import torch
print(f"[env] torch={torch.__version__} cuda={torch.cuda.is_available()} gpu={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")
print(f"[env] numpy={np.__version__}")

# Smoke-test the heavy imports so you fail here, not 30 minutes into training.
import importlib
for mod in ("numpy","scipy","librosa","soundfile","wandb","torchaudio",
            "asteroid_filterbanks","asteroid","auraloss","onnx","onnxruntime","tqdm"):
    try:
        importlib.import_module(mod)
        print(f"  ok  {mod}")
    except Exception as e:
        print(f"  FAIL {mod}: {e}")

In [ ]:
# Stage datasets onto local disk for fast I/O during training.
# Tarballs live in bridge/datasets/; we untar each into /content/data/<name>/.
# The training config's dataset paths get rewritten to /content/data/... in the
# next cell, so nothing else needs to know where these live.
#
# Tar layout assumption: each tarball's top-level dir equals its stem (so
# syn_1_5m.tar contains syn_1_5m/..., producing /content/data/syn_1_5m/syn_1_5m/
# after extraction into /content/data/syn_1_5m/). This matches the existing
# repo configs. If your tarballs are flat, drop the inner name in their paths.
TAR_MAP = {
    "syn_1m.tar":    "syn_1m",
    "syn_1_5m.tar":  "syn_1_5m",
    "syn_2m.tar":    "syn_2m",
    "syn_test.tar":  "syn_test",
}
DATA_ROOT = "/content/data"
!mkdir -p "$DATA_ROOT"

import os, subprocess, time
for tar, d in TAR_MAP.items():
    dest = os.path.join(DATA_ROOT, d)
    if os.path.isdir(dest) and os.listdir(dest):
        print(f"[data] {d}: already staged ({len(os.listdir(dest))} entries)")
        continue
    src = os.path.join(BRIDGE, "datasets", tar)
    if not os.path.isfile(src):
        print(f"[data] SKIP {tar}: not found at {src} (move tarball into bridge/datasets/)")
        continue
    os.makedirs(dest, exist_ok=True)
    t0 = time.time()
    print(f"[data] extracting {src} -> {dest}")
    subprocess.run(["tar", "-xf", src, "-C", dest], check=True)
    print(f"[data]   done in {time.time()-t0:.1f}s")

print("[data] staged contents:")
!ls -la "$DATA_ROOT"

In [ ]:
# Resolve run_dir — bridge/runs/EXP_NAME. If it already exists and
# RESUME_FROM_DRIVE is set, training picks up from checkpoints/best.pt.
import os, json, re, shutil

BRIDGE_RUN = os.path.join(BRIDGE, "runs", EXP_NAME)
LOCAL_RUN  = os.path.join("/content/runs", EXP_NAME)
os.makedirs(BRIDGE_RUN, exist_ok=True)
os.makedirs(os.path.dirname(LOCAL_RUN), exist_ok=True)

# Copy bridge run into local disk (faster checkpoint I/O during training).
if os.path.exists(LOCAL_RUN):
    shutil.rmtree(LOCAL_RUN)
if RESUME_FROM_DRIVE and os.path.isdir(os.path.join(BRIDGE_RUN, "checkpoints")):
    print(f"[run] resuming from bridge run: {BRIDGE_RUN}")
    shutil.copytree(BRIDGE_RUN, LOCAL_RUN)
else:
    os.makedirs(LOCAL_RUN)
    print(f"[run] fresh run: {LOCAL_RUN}")

# configs live in the repo root snapshot under REPO_ROOT.
CONFIG_PATH = os.path.join(REPO_ROOT, "real_experiments", CONFIG_NAME)
assert os.path.isfile(CONFIG_PATH), f"config not found: {CONFIG_PATH}"

# Rewrite two things in-place before training:
#   (a) hardcoded Windows dataset paths -> Colab local-disk paths.
#   (b) epochs -> EPOCHS_TARGET (train_pt.py reads params['epochs']).
# Leaves the original repo file touched (which is fine, it lives inside the
# local /content copy; the bridge's pristine copy is untouched).
with open(CONFIG_PATH) as f:
    cfg_text = f.read()

# Path fix: anything rooted at ".../Sound_Bubble/datasets/..." becomes
# "/content/data/...". Works for C:/..., /home/..., or any other prefix.
cfg_text_new = re.sub(
    r'["\'][A-Za-z]?:?[\\/][^"\']*?[\\/]Sound_Bubble[\\/]datasets[\\/]',
    '"/content/data/',
    cfg_text,
)
# Also catch any bare "datasets/" relative paths if someone edited the config
# without the full prefix.
cfg_text_new = re.sub(
    r'"datasets[\\/]',
    '"/content/data/',
    cfg_text_new,
)
paths_changed = cfg_text_new != cfg_text
cfg = json.loads(cfg_text_new)

cfg_epochs_before = cfg.get("epochs")
if EPOCHS_TARGET is not None and EPOCHS_TARGET != cfg_epochs_before:
    cfg["epochs"] = int(EPOCHS_TARGET)

with open(CONFIG_PATH, "w") as f:
    json.dump(cfg, f, indent=4)

print(f"[cfg] dataset-path rewrite: {'yes' if paths_changed else 'no (already portable)'}")
print(f"[cfg] epochs: {cfg_epochs_before} -> {cfg.get('epochs')}")
print(f"[cfg] config={CONFIG_PATH}")
print(f"[cfg] local_run={LOCAL_RUN}")
print(f"[cfg] bridge_run={BRIDGE_RUN}")

# Sanity: confirm at least one dataset dir the config now points at actually exists.
sample_dirs = []
for k in ("train_data_args", "val_data_args", "test_data_args"):
    for entry in (cfg.get(k) or {}).get("dataset_dirs") or []:
        p = entry.get("path")
        if p:
            sample_dirs.append(p)
missing = [p for p in sample_dirs if not os.path.isdir(p)]
if missing:
    print("[cfg] WARNING — these dataset dirs do not exist on Colab:")
    for p in missing:
        print(f"  - {p}")
    print("[cfg] Did you run the dataset-staging cell and put the tarballs in bridge/datasets/?")
else:
    print(f"[cfg] all {len(sample_dirs)} dataset dirs resolve on disk")


In [ ]:
# Kick off training. Re-run this cell to resume after a disconnect — it'll
# pick up from checkpoints/best.pt inside LOCAL_RUN (train_pt.py handles resume).
#
# Avoid `!` interpolation edge-cases by launching via subprocess.
import os, sys, subprocess, glob, textwrap

# ---- Preflight: checkpoints + stale bridge/code detection ----
def _ls_ckpts(label, run_dir):
    ck = os.path.join(run_dir, "checkpoints")
    print(f"\n[preflight][{label}] {ck}")
    if not os.path.isdir(ck):
        print("  (missing checkpoints/ dir)")
        return
    pts = sorted(glob.glob(os.path.join(ck, "*.pt")))
    if not pts:
        print("  (no .pt files)")
        return
    for p in pts:
        try:
            sz = os.path.getsize(p)
        except OSError:
            sz = -1
        print(f"  - {os.path.basename(p)} ({sz} bytes)")

_ls_ckpts("LOCAL_RUN", LOCAL_RUN)
_ls_ckpts("BRIDGE_RUN", BRIDGE_RUN)

sp = os.path.join(REPO_ROOT, "src", "datasets", "perturbations", "SpeedPerturbation.py")
print("\n[preflight] SpeedPerturbation.py head:")
print("-----")
if os.path.isfile(sp):
    with open(sp, "r", encoding="utf-8", errors="replace") as f:
        print("".join(f.readlines()[:25]), end="")
else:
    print(f"MISSING: {sp}")
print("-----")

# Hot-patch SpeedPerturbation if the staged bridge snapshot is stale (Colab torchaudio often has no sox_effects).
if os.path.isfile(sp):
    s = open(sp, "r", encoding="utf-8", errors="replace").read()
    if "hasattr(torchaudio, \"sox_effects\")" not in s:
        print("\n[preflight][patch] Writing Colab-safe SpeedPerturbation.py into REPO_ROOT...")
        open(sp, "w", encoding="utf-8").write(textwrap.dedent("""\
import torch
import torchaudio
import torch.nn.functional as F

class SpeedPerturbation:
    def __init__(self, min_speed, max_speed, sample_rate = 24000):
        self.min_speed = min_speed
        self.max_speed = max_speed
        self.sample_rate = sample_rate

    def __call__(self, audio_data, gt_audio):
        T = audio_data.shape[-1]
        speed_factor = torch.rand((1,)).item() * (self.max_speed - self.min_speed) + self.min_speed

        if hasattr(torchaudio, "sox_effects"):
            try:
                sox_effects = [
                    ["speed", str(speed_factor)],
                    ["rate", str(self.sample_rate)],
                ]
                transformed_audio, _ = torchaudio.sox_effects.apply_effects_tensor(
                    audio_data, self.sample_rate, sox_effects
                )
                gt_audio, _ = torchaudio.sox_effects.apply_effects_tensor(
                    gt_audio, self.sample_rate, sox_effects
                )
            except Exception:
                transformed_audio = self._speed_fallback(audio_data, speed_factor)
                gt_audio = self._speed_fallback(gt_audio, speed_factor)
        else:
            transformed_audio = self._speed_fallback(audio_data, speed_factor)
            gt_audio = self._speed_fallback(gt_audio, speed_factor)

        if transformed_audio.shape[-1] > T:
            transformed_audio = transformed_audio[..., :T]
            gt_audio = gt_audio[..., :T]
        else:
            transformed_audio = F.pad(transformed_audio, (0, T - transformed_audio.shape[-1]))
            gt_audio = F.pad(gt_audio, (0, T - gt_audio.shape[-1]))

        assert transformed_audio.shape[-1] == T
        assert gt_audio.shape[-1] == T
        return transformed_audio, gt_audio

    @staticmethod
    def _speed_fallback(waveform: torch.Tensor, speed_factor: float) -> torch.Tensor:
        t_in = waveform.shape[-1]
        t_out = max(1, int(round(t_in / speed_factor)))
        return F.interpolate(
            waveform.unsqueeze(0),
            size=t_out,
            mode="linear",
            align_corners=False,
        ).squeeze(0)
"""))
    else:
        print("\n[preflight] SpeedPerturbation.py already looks Colab-safe.")

# Hot-patch train_pt.py if the bridge snapshot is missing best.pt resume + error propagation.
tp = os.path.join(REPO_ROOT, "src", "train_pt.py")
if os.path.isfile(tp):
    t = open(tp, "r", encoding="utf-8", errors="replace").read()
    needs_best_resume = ("elif os.path.exists(best_path):" not in t) and ("checkpoints/best.pt" not in t)
    needs_raise = ("except Exception as _:" in t) and ("traceback.print_exc()" in t) and ("raise" not in t.split("traceback.print_exc()", 1)[1].split("if __name__", 1)[0])

    if needs_best_resume or needs_raise:
        print("\n[preflight][patch] Updating train_pt.py for Colab (best.pt resume + fail-fast)...")

        if needs_best_resume:
            needle = "    if os.path.exists(state_path):\n        hl_module.load_state(state_path)\n\n    start_epoch = hl_module.epoch\n"
            insert = (
                "    if os.path.exists(state_path):\n"
                "        print(f\"[resume] loading {state_path}\")\n"
                "        hl_module.load_state(state_path)\n"
                "    elif os.path.exists(best_path):\n"
                "        print(f\"[resume] checkpoints/last.pt missing; loading {best_path}\")\n"
                "        hl_module.load_state(best_path)\n"
                "    else:\n"
                "        print(\"[resume] no checkpoints/last.pt or checkpoints/best.pt found; starting from epoch 0\")\n\n"
                "    start_epoch = hl_module.epoch\n"
            )
            if needle in t:
                t = t.replace(needle, insert)
            else:
                print("[preflight][patch] WARNING: could not auto-patch resume block (unexpected train_pt.py layout).")

        if needs_raise:
            t = t.replace(
                "    except Exception as _:\n        import traceback\n        traceback.print_exc()\n",
                "    except Exception as _:\n        import traceback\n        traceback.print_exc()\n        raise\n",
            )

        open(tp, "w", encoding="utf-8").write(t)
    else:
        print("\n[preflight] train_pt.py already includes Colab resume/fail-fast fixes.")

os.makedirs(LOCAL_RUN, exist_ok=True)
log_path = os.path.join(LOCAL_RUN, "colab_train.log")

env = os.environ.copy()
env["PYTHONPATH"] = REPO_ROOT + ":" + env.get("PYTHONPATH", "")

cmd = [
    sys.executable, "-m", "src.train_pt",
    "--config", CONFIG_PATH,
    "--run_dir", LOCAL_RUN,
    "--project_name", "sound_bubble_colab",
    "--no_wandb",
]
print("[train]", " ".join(cmd))
print(f"[train] log={log_path}")

with open(log_path, "a", buffering=1) as f:
    p = subprocess.Popen(
        cmd,
        cwd=REPO_ROOT,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert p.stdout is not None
    for line in p.stdout:
        print(line, end="")
        f.write(line)

rc = p.wait()
print(f"[train] exit_code={rc}")
if rc != 0:
    raise RuntimeError(f"train_pt failed with exit code {rc}")

In [ ]:
# Push the run back to the bridge so you can pull from the PC side later,
# and so the next notebook run can resume from here.
import subprocess, os
print(f"[sync] {LOCAL_RUN} -> {BRIDGE_RUN}")
subprocess.run(["rsync", "-a", "--delete", LOCAL_RUN + "/", BRIDGE_RUN + "/"], check=True)
print("[sync] run mirrored to bridge")
!du -sh "$BRIDGE_RUN"

In [ ]:
# Export ONNX into the bridge zoo. Provenance fields (radius, epochs,
# source_config, source_run_dir, source_checkpoint) are recorded in the
# paired runtime.json automatically.
import os, subprocess, sys
BRIDGE_ZOO = os.path.join(BRIDGE, "zoo")
os.makedirs(BRIDGE_ZOO, exist_ok=True)

ONNX_NAME    = f"model_{RADIUS_TAG}.onnx"
ONNX_OUT     = os.path.join(BRIDGE_ZOO, ONNX_NAME)
CONTRACT_OUT = os.path.join(BRIDGE_ZOO, ONNX_NAME.replace(".onnx", ".runtime.json"))

env = os.environ.copy()
env["PYTHONPATH"] = REPO_ROOT + ":" + env.get("PYTHONPATH", "")

cmd = [
    sys.executable, "edge/export_to_onnx.py",
    "--run-dir", LOCAL_RUN,
    "--output", ONNX_OUT,
    "--contract-out", CONTRACT_OUT,
]
print("[export]", " ".join(cmd))
res = subprocess.run(cmd, cwd=REPO_ROOT, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
# Keep notebook output compact, like the old `tail -30`
print("\n".join(res.stdout.splitlines()[-30:]))
res.check_returncode()

print("\n[zoo] bridge zoo contents:")
print(subprocess.run(["bash", "-lc", f"ls -la '{BRIDGE_ZOO}'"], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT).stdout)

---

## Done — back on the PC

```powershell
cd C:\Users\darre\Sound_Bubble
.\colab\sync_bridge_to_workspace.ps1            # pulls edge/zoo/
.\colab\sync_bridge_to_workspace.ps1 -PullRuns  # also copies runs/ back (heavy)
```

The live pipeline now resolves the new radius via:

```
python edge/inference_pipeline.py --zoo-dir edge/zoo --bubble-radius 1.5 ...
```

### Iterating

- To train a different radius: change `RADIUS` in the first cell, re-run all.
- To extend an existing radius's training: keep `RESUME_FROM_DRIVE = True`, bump `EPOCHS_TARGET`, re-run all.
- To start fresh: set `RESUME_FROM_DRIVE = False` or use a new `EXPERIMENT_TAG`.